In [ ]:
!pip install openai==0.28

In [ ]:
!pip install rank_bm25

# Importing Dependencies

In [3]:
import openai
import pandas as pd
import numpy as np
import re
import time
import random

In [4]:
import json

In [ ]:
openai.api_key = "API KEY"

# Datasets

In [6]:
df = pd.read_csv("../Code_Snippets.csv")

In [7]:
df.head()

,ID,Problem Title,Problem Description,Submitted Code,Problem Type,Source
0,1,Find All K-Distant Indices in an Array,You are given a 0-indexed integer array nums a...,class Solution:\r\n def findKDistantIndices...,Easy,LeetCode
1,2,Two Sum,Given an array of integers nums and an integer...,"class Solution:\r\n def twoSum(self, nums: ...",Easy,LeetCode
2,3,Symmetric Tree,"Given the root of a binary tree, check whether...","class Solution:\r\n def isSymmetric(self, r...",Easy,LeetCode
3,4,Same Tree,"Given the roots of two binary trees p and q, w...","class Solution:\r\n def isSameTree(self, p:...",Easy,LeetCode
4,5,Binary Tree Inorder Traversal,"Given the root of a binary tree, return the in...",class Solution:\r\n def inorderTraversal(se...,Easy,LeetCode


In [8]:
df.shape

(300, 6)

In [9]:
human_eval_df = pd.read_csv("../rag_data/test.csv")

In [10]:
human_eval_df.shape

(164, 5)

In [11]:
file_path = '../rag_data/mbpp.jsonl'
data = []
with open(file_path, 'r') as file:
    for line in file:
        data.append(json.loads(line))

mbpp_df = pd.DataFrame(data)
mbpp_df.shape

(974, 6)

In [12]:
task_descriptions = (df['Problem Title'] + " " + df['Problem Description']).tolist()
code_snippets = df['Submitted Code'].tolist()

# ZeroShot Prompt 1

In [13]:
MAX_RETRIES = 7
BASE_DELAY = 7
start_index = 0
data = []

In [14]:
from rank_bm25 import BM25Okapi
import math

class BM25L(BM25Okapi):
    def __init__(self, corpus, k1=1.5, b=0.75, delta=0.5):
        super().__init__(corpus, k1=k1, b=b)
        self.delta = delta  # extra term in BM25L

    def get_scores(self, query):
        scores = [0.0] * len(self.doc_freqs)
        for q in query:
            if q not in self.idf:
                continue
            q_idf = self.idf[q]
            for index, freq in enumerate(self.doc_freqs):
                f = freq.get(q, 0)
                dl = self.doc_len[index]
                numerator = f + self.delta
                denominator = self.k1 * ((1 - self.b) + self.b * dl / self.avgdl) + f
                score = q_idf * numerator / denominator
                scores[index] += score
        return scores

In [ ]:
!pip install tiktoken

In [37]:
import faiss
from sentence_transformers import SentenceTransformer

In [38]:
human_eval_df["prompt"] = human_eval_df["prompt"].astype(str)
human_eval_df["canonical_solution"] = human_eval_df["canonical_solution"].astype(str)
human_eval_df["test"] = human_eval_df["test"].astype(str)

In [39]:
mbpp_df["text"] = mbpp_df["text"].astype(str)
mbpp_df["code"] = mbpp_df["code"].astype(str)
mbpp_df["test_list"] = mbpp_df["test_list"].astype(str)

In [40]:
human_eval_texts = human_eval_df["prompt"] + " " + human_eval_df["canonical_solution"] + " " + human_eval_df["test"]
mbpp_texts = mbpp_df["text"] + " " + mbpp_df["code"] + " " + mbpp_df["test_list"]

In [41]:
all_texts = list(human_eval_texts) + list(mbpp_texts)

In [42]:
# Load embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [43]:
embeddings = model.encode(all_texts, convert_to_numpy=True)

In [44]:
# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save index
faiss.write_index(index, "faiss_index.bin")

# Save mapping of text to index
df_mapping = pd.DataFrame({"text": all_texts})
df_mapping.to_csv("index_mapping.csv", index=False)

In [45]:
# Function to retrieve relevant examples
def retrieve_examples(query, top_k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = [df_mapping.iloc[i]["text"] for i in indices[0]]
    return results

In [46]:
data2 = []

In [47]:
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
    retries = 0
    success = False

    # Retrieve similar examples using FAISS
    retrieved_examples = retrieve_examples(task_description, top_k=3)
    retrieved_text = "\n\n".join(retrieved_examples)

    while retries < MAX_RETRIES:
        try:
            # Generate the response for the code snippet using chat-completions endpoint
            response = openai.ChatCompletion.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": f"""You are a helpful AI assistant working with developers to create unit test cases
                        from codes to help the development process. You have access to a retrieval system that provides
                        relevant past examples to assist in generating high-quality test cases."""
                    },
                    {
                        "role": "user",
                        "content": f"""You are given a Python function along with its task description. Generate a complete
                        and executable set of test cases for the function using Python.
                        Task Description: {task_description}
                        Code: {code_snippet}
                        Here are some relevant test case examples retrieved from a similar problem to improve quality:
                        {retrieved_text}
                        Test Script: [generate ONLY the test script]
                        """
                    }
                ],
                temperature=0.7
            )
            data2.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
            print(f"{i}. Processed successfully")
            print(response['choices'][0]['message']['content'].strip())
            time.sleep(BASE_DELAY)  # Apply base delay after success
            success = True
            break

        except Exception as e:
            print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
            retries += 1
            if "429" in str(e):  # Rate limit error
                delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
                print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
            else:
                break  # Break for non-rate-limit errors

    if not success:
        # Append fallback response if all retries fail
        data2.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
        print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

1. Processed successfully
```python
import unittest

class TestFindKDistantIndices(unittest.TestCase):
    def setUp(self):
        self.solution = Solution()

    def test_example_1(self):
        nums = [3, 4, 9, 1, 3, 9, 5]
        key = 9
        k = 1
        expected = [1, 2, 3, 4, 5, 6]
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_example_2(self):
        nums = [2, 2, 2, 2, 2]
        key = 2
        k = 2
        expected = [0, 1, 2, 3, 4]
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_no_k_distant_indices(self):
        nums = [1, 2, 3, 4, 5]
        key = 6
        k = 1
        expected = []
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_all_elements_are_key(self):
        nums = [5, 5, 5, 5, 5]
        key = 5
        k = 0
        expected = [0, 1, 

In [48]:
output_df2 = pd.DataFrame(data2)
output_df2.to_csv("FAISS_Zerpshot_Prompt1.csv", index=False)
print("Test cases saved to 'FAISS_Zerpshot_Prompt1.csv'")

Test cases saved to 'FAISS_Zerpshot_Prompt1.csv'


In [15]:
from rank_bm25 import BM25Okapi

In [16]:
human_eval_df["prompt"] = human_eval_df["prompt"].astype(str)
human_eval_df["canonical_solution"] = human_eval_df["canonical_solution"].astype(str)
human_eval_df["test"] = human_eval_df["test"].astype(str)

mbpp_df["text"] = mbpp_df["text"].astype(str)
mbpp_df["code"] = mbpp_df["code"].astype(str)
mbpp_df["test_list"] = mbpp_df["test_list"].astype(str)

In [17]:
all_tasks = human_eval_df['prompt'].tolist() + mbpp_df['text'].tolist()
all_codes = human_eval_df['canonical_solution'].tolist() + mbpp_df['code'].tolist()
all_tests = human_eval_df['test'].tolist() + mbpp_df['test_list'].tolist()

In [18]:
# Tokenize text for BM25L
tokenized_corpus = [task.split() for task in all_tasks]

In [19]:
bm25 = BM25L(tokenized_corpus, k1=1.5, b=0.75, delta=0.5)

In [20]:
def retrieve_examples(task_description, top_k=3):
    tokenized_query = task_description.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    retrieved = [f"Task: {all_tasks[i]}\nCode: {all_codes[i]}\nTest: {all_tests[i]}" for i in top_indices]
    return retrieved

In [21]:
data3 = []
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
    retries = 0
    success = False

    # Retrieve examples using BM25
    retrieved_examples = retrieve_examples(task_description, top_k=3)
    retrieved_text = "\n\n".join(retrieved_examples)

    while retries < MAX_RETRIES:
        try:
            # Generate response using OpenAI API
            response = openai.ChatCompletion.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": """You are a helpful AI assistant working with developers to create unit test cases from codes
                        to help the development process. You have access to a retrieval system that provides relevant past examples
                        to assist in generating high-quality test cases."""
                    },
                    {
                        "role": "user",
                        "content": f"""You are given a Python function along with its task description. Generate a complete
                        and executable set of test cases for the function using Python.
                        Task Description: {task_description}
                        Code: {code_snippet}
                        Here are some relevant test case examples retrieved from a similar problem to improve quality:
                        {retrieved_text}
                        Test Script: [generate ONLY the test script]
                        """
                    }
                ],
                temperature=0.7
            )
            data3.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
            print(f"{i}. Processed successfully")
            print(response['choices'][0]['message']['content'].strip())
            time.sleep(BASE_DELAY)  # Apply base delay after success
            success = True
            break

        except Exception as e:
            print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
            retries += 1
            if "429" in str(e):  # Rate limit error
                delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
                print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
            else:
                break  # Break for non-rate-limit errors

    if not success:
        data3.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
        print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

1. Processed successfully
```python
import unittest

class TestFindKDistantIndices(unittest.TestCase):
    def setUp(self):
        self.solution = Solution()

    def test_example_1(self):
        nums = [3, 4, 9, 1, 3, 9, 5]
        key = 9
        k = 1
        expected = [1, 2, 3, 4, 5, 6]
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_example_2(self):
        nums = [2, 2, 2, 2, 2]
        key = 2
        k = 2
        expected = [0, 1, 2, 3, 4]
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_single_element_array(self):
        nums = [1]
        key = 1
        k = 0
        expected = [0]
        result = self.solution.findKDistantIndices(nums, key, k)
        self.assertEqual(result, expected)

    def test_no_k_distant_indices(self):
        nums = [1, 3, 5, 7, 9]
        key = 2
        k = 1
        expected = []
        result 

In [ ]:
output_df3 = pd.DataFrame(data3)
output_df3.to_csv("BM25L_Zeroshot_Prompt1.csv", index=False)
print("Test cases saved to 'BM25L_Zeroshot_Prompt1.csv'")